# Advanced filtering


## Refresh basic boolean filtering

Use the Titanic dataset again

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Select adult passengers (Age > 18).

Note:
- if you intend to modify the filtered dataframe, make sure you make a copy, e.g., `adults = df[df['age'] > 10].copy()`.

In [15]:
adults = df[df["Age"] >= 18]

adults[["Name","Age","Sex"]].head()


,Name,Age,Sex
0,"Braund, Mr. Owen Harris",22.0,male
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,female
2,"Heikkinen, Miss. Laina",26.0,female
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,female
4,"Allen, Mr. William Henry",35.0,male


In [16]:
df['Age'] >= 18

0       True
1       True
2       True
3       True
4       True
       ...  
886     True
887     True
888    False
889     True
890     True
Name: Age, Length: 891, dtype: bool

### Alternatives: using Pandas built-in comparison methods

The same filters can be achieved by using pandas methods:

- `.gt()`: greater than
- `.lt()`: less than
- `.ge()`: greater than or equal to
- `.le()`: less than or equal to

Note: the advantage of those methods is for more complex pipeline or method chaining.


In [17]:
adults = df[df["Age"].ge(18)]

adults[["Name","Age","Sex"]].head()

,Name,Age,Sex
0,"Braund, Mr. Owen Harris",22.0,male
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,female
2,"Heikkinen, Miss. Laina",26.0,female
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,female
4,"Allen, Mr. William Henry",35.0,male


or, if we wanted to filter children.


In [18]:
children = df[df['Age'].lt(18)]

children[["Name","Age","Sex"]].head()

,Name,Age,Sex
7,"Palsson, Master. Gosta Leonard",2.0,male
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,female
10,"Sandstrom, Miss. Marguerite Rut",4.0,female
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0,female
16,"Rice, Master. Eugene",2.0,male


### Filtering with Multiple Conditions
#### "AND" conditions: `&`
Example: female passengers in first class

Important rule: Each condition must be inside parentheses.

In [11]:
first_class_women = df[
    (df["Sex"] == "female") & (df["Pclass"] == 1)
]

first_class_women[["Name","Sex","Pclass"]].head()


,Name,Sex,Pclass
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,1
11,"Bonnell, Miss. Elizabeth",female,1
31,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,1
52,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,1


#### "OR" conditions: `|`

Example: passengers who are children OR elderly

In [12]:
young_or_old = df[
    (df["Age"] < 18) | (df["Age"] > 60)
]

young_or_old[["Name","Age"]].head()


,Name,Age
7,"Palsson, Master. Gosta Leonard",2.0
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0
10,"Sandstrom, Miss. Marguerite Rut",4.0
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0
16,"Rice, Master. Eugene",2.0


### Negation of conditions

Example: Select passengers who are NOT children.

Important Note: the difference between `df[df["Age"] >= 18)]` and `df[~(df["Age"] < 18)]` is that the former only select passengers whose age is equal to or greater than 18, but the latter will also select NaN values.

In [34]:
not_children = df[~(df["Age"] < 18)]

not_children[["Name","Age"]].head()

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
2,"Heikkinen, Miss. Laina",26.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0
4,"Allen, Mr. William Henry",35.0


### Practical Examples
#### Example #1

Find third-class adult male passengers with very low fare.

In [35]:
low_fare_passengers = df[
    (df["Pclass"] == 3) &
    (df["Sex"] == "male") &
    (df["Fare"] < 10) &
    (~(df["Age"] < 18)) # You can use df['Age'] >= 18 but here is just a nice exercise
]

low_fare_passengers[["Name","Pclass","Sex", "Age", "Fare"]].head()


,Name,Pclass,Sex,Age,Fare
0,"Braund, Mr. Owen Harris",3,male,22.0,7.2500
4,"Allen, Mr. William Henry",3,male,35.0,8.0500
5,"Moran, Mr. James",3,male,NaN,8.4583
12,"Saundercock, Mr. William Henry",3,male,20.0,8.0500
26,"Emir, Mr. Farred Chehab",3,male,NaN,7.2250


#### Example #2
Check survival rate of children vs adults.

In [42]:
children = df[df["Age"] < 18]
adults = df[df["Age"] >= 18]

print(f"Children survival rate: {children['Survived'].mean():.2f}")
print(f"Adult survival rate: {adults['Survived'].mean():.2f}")


Children survival rate: 0.54
Adult survival rate: 0.38


## Advanced Filtering
### Check whether values belong to a set of categories

Use `.isin()`

Example: Select passengers from Southampton or Cherbourg.

In the Titanic dataset, the `Embarked` column indicates the port where a passenger boarded, categorized as C (Cherbourg), Q (Queenstown), or S (Southampton).

In [43]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../06-Exploratory_data_analysis/data/Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [51]:
df[df["Embarked"].isin(["S", "C"])][["Name", "Embarked"]]


,Name,Embarked
0,"Braund, Mr. Owen Harris",S
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C
2,"Heikkinen, Miss. Laina",S
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S
4,"Allen, Mr. William Henry",S
...,...,...
884,"Sutehall, Mr. Henry Jr",S
886,"Montvila, Rev. Juozas",S
887,"Graham, Miss. Margaret Edith",S
888,"Johnston, Miss. Catherine Helen ""Carrie""",S


Or, if we wanted to select passengers who (1) boarded at Southampton or Cherbourg, and (2) in the first or second class.

In [50]:
df[
    (df['Embarked'].isin(['S', 'C'])) &
    (df['Pclass'].isin([1, 2]))
][["Name", "Embarked",'Pclass']]

,Name,Embarked,Pclass
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,1
6,"McCarthy, Mr. Timothy J",S,1
9,"Nasser, Mrs. Nicholas (Adele Achem)",C,2
11,"Bonnell, Miss. Elizabeth",S,1
...,...,...,...
880,"Shelley, Mrs. William (Imanita Parrish Hall)",S,2
883,"Banfield, Mr. Frederick James",S,2
886,"Montvila, Rev. Juozas",S,2
887,"Graham, Miss. Margaret Edith",S,1


### Filtering by Range: `.between()`

Example: passengers aged between 1 and 4, inclusive.

In [56]:
df[df["Age"].between(left=1, right=3, inclusive="both")][["Name","Age"]]


,Name,Age
7,"Palsson, Master. Gosta Leonard",2.0
16,"Rice, Master. Eugene",2.0
43,"Laroche, Miss. Simonne Marie Anne Andree",3.0
119,"Andersson, Miss. Ellis Anna Maria",2.0
164,"Panula, Master. Eino Viljami",1.0
172,"Johnson, Miss. Eleanor Ileen",1.0
183,"Becker, Master. Richard F",1.0
193,"Navratil, Master. Michel M",3.0
205,"Strom, Miss. Telma Matilda",2.0
261,"Asplund, Master. Edvin Rojj Felix",3.0


The code above is equivalent to `(df["Age"] >= 1) & (df["Age"] <= 4)` but easy to read.

Note: `.between()` can also pass strings as left or right values. In this case, the strings are compared lexicographically (in alphabetical order). example:

In [58]:
df[df['Name'].between(left="Alice", right="Bob")][['Name']]

,Name
4,"Allen, Mr. William Henry"
13,"Andersson, Mr. Anders Johan"
21,"Beesley, Mr. Lawrence"
25,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia..."
49,"Arnold-Franchi, Mrs. Josef (Josefine Franchi)"
...,...
858,"Baclini, Mrs. Solomon (Latifa Qurban)"
870,"Balkic, Mr. Cerin"
871,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)"
883,"Banfield, Mr. Frederick James"


### Filtering Using String Methods

These operate on text columns.

#### `.str.contains()`

Example: passengers whose name contains "Smith".

Note that the difference between setting `case=True` and `case=False`.

In [61]:
df[df["Name"].str.contains("Smith", na=False, case=False)][["Name"]]


,Name
165,"Goldsmith, Master. Frank John William ""Frankie"""
174,"Smith, Mr. James Clinch"
260,"Smith, Mr. Thomas"
284,"Smith, Mr. Richard William"
328,"Goldsmith, Mrs. Frank John (Emily Alice Brown)"
346,"Smith, Miss. Marion Elsie"
548,"Goldsmith, Mr. Frank John"


#### `.str.startswith()`
Example: Passengers whose name starts with "Smith".

In [63]:
df[df["Name"].str.startswith("Smith", na=False)][["Name"]]


,Name
174,"Smith, Mr. James Clinch"
260,"Smith, Mr. Thomas"
284,"Smith, Mr. Richard William"
346,"Smith, Miss. Marion Elsie"


#### `.str.endswith()`

Example: passengers whose cabin ends with "5".

In [65]:
df[df["Cabin"].str.endswith("5", na=False)][["Name","Cabin"]]


,Name,Cabin
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C85
96,"Goldschmidt, Mr. George B",A5
218,"Bazzani, Miss. Albina",D15
248,"Beckwith, Mr. Richard Leonard",D35
268,"Graham, Mrs. William Thompson (Edith Junkins)",C125
307,"Penasco y Castellana, Mrs. Victor de Satode (M...",C65
369,"Aubart, Mme. Leontine Pauline",B35
505,"Penasco y Castellana, Mr. Victor de Satode",C65
512,"McGough, Mr. James Robert",E25
527,"Farthing, Mr. John",C95


#### `.str.isnumeric()`
Check whether a string column contains only numbers.

Example: numeric ticket numbers.

In [66]:
df[df["Ticket"].str.isnumeric()][["Ticket","Name"]].head()


,Ticket,Name
3,113803,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,373450,"Allen, Mr. William Henry"
5,330877,"Moran, Mr. James"
6,17463,"McCarthy, Mr. Timothy J"
7,349909,"Palsson, Master. Gosta Leonard"


### `.nlargest()` and `.nsmallest()`

Quickly find top or bottom values without sorting the entire dataframe.

In [67]:
# Top 5 most expensive tickets.

df.nlargest(5, "Fare")[["Name","Fare","Pclass"]]


,Name,Fare,Pclass
258,"Ward, Miss. Anna",512.3292,1
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1
737,"Lesurer, Mr. Gustave J",512.3292,1
27,"Fortune, Mr. Charles Alexander",263.0000,1
88,"Fortune, Miss. Mabel Helen",263.0000,1


In [68]:
# Smallest fares

df.nsmallest(5, "Fare")[["Name","Fare","Pclass"]]


,Name,Fare,Pclass
179,"Leonard, Mr. Lionel",0.0,3
263,"Harrison, Mr. William",0.0,1
271,"Tornquist, Mr. William Henry",0.0,3
277,"Parkes, Mr. Francis ""Frank""",0.0,2
302,"Johnson, Mr. William Cahoone Jr",0.0,3


### `.any()` and `.all()`

- `axis=1`: evaluate across columns for each row
- `axis=0`: evaluate across rows for each columns

#### `.any()`
Example: find passengers where any family member is present (either SibSp or Parch > 0).

In [70]:
# first, we can create a condition called family_condition, which is a Dataframe with two Series with True or False values

family_condition = df[["SibSp", "Parch"]] > 0

family_condition

,SibSp,Parch
0,True,False
1,True,False
2,False,False
3,True,False
4,False,False
...,...,...
886,False,False
887,False,False
888,True,True
889,False,False


In [72]:
# Now check if any column is True for each row.

passengers_with_any_family = df[family_condition.any(axis=1)][["Name","SibSp","Parch"]]

passengers_with_any_family

,Name,SibSp,Parch
0,"Braund, Mr. Owen Harris",1,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0
7,"Palsson, Master. Gosta Leonard",3,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",0,2
...,...,...,...
874,"Abelson, Mrs. Samuel (Hannah Wizosky)",1,0
879,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",0,1
880,"Shelley, Mrs. William (Imanita Parrish Hall)",0,1
885,"Rice, Mrs. William (Margaret Norton)",0,5


In [73]:
# The above code is equivalent to the following code:

passengers_with_any_family = df[(df[["SibSp", "Parch"]]>0).any(axis=1)][["Name","SibSp","Parch"]]

passengers_with_any_family

,Name,SibSp,Parch
0,"Braund, Mr. Owen Harris",1,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0
7,"Palsson, Master. Gosta Leonard",3,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",0,2
...,...,...,...
874,"Abelson, Mrs. Samuel (Hannah Wizosky)",1,0
879,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",0,1
880,"Shelley, Mrs. William (Imanita Parrish Hall)",0,1
885,"Rice, Mrs. William (Margaret Norton)",0,5


In [75]:
# Also equivalent to the following code (without using .any():

passengers_with_any_family = df[((df['SibSp'] > 0) | (df['Parch'] > 0))][["Name","SibSp","Parch"]]

passengers_with_any_family

,Name,SibSp,Parch
0,"Braund, Mr. Owen Harris",1,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0
7,"Palsson, Master. Gosta Leonard",3,1
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",0,2
...,...,...,...
874,"Abelson, Mrs. Samuel (Hannah Wizosky)",1,0
879,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",0,1
880,"Shelley, Mrs. William (Imanita Parrish Hall)",0,1
885,"Rice, Mrs. William (Margaret Norton)",0,5


#### `.all()`
Example: find passengers who have both siblings/spouses AND parents/children onboard.

In [82]:
family_condition = df[["SibSp", "Parch"]] > 0

df[family_condition.all(axis=1)][["Name","SibSp","Parch"]]


,Name,SibSp,Parch
7,"Palsson, Master. Gosta Leonard",3,1
10,"Sandstrom, Miss. Marguerite Rut",1,1
13,"Andersson, Mr. Anders Johan",1,5
16,"Rice, Master. Eugene",4,1
24,"Palsson, Miss. Torborg Danira",3,1
...,...,...,...
856,"Wick, Mrs. George Dennick (Mary Hitchcock)",1,1
863,"Sage, Miss. Dorothy Edith ""Dolly""",8,2
869,"Johnson, Master. Harold Theodor",1,1
871,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",1,1


In [78]:
# this is equivalent to

df[(df['SibSp'] > 0) & (df['Parch'] > 0)][["Name","SibSp","Parch"]]

,Name,SibSp,Parch
7,"Palsson, Master. Gosta Leonard",3,1
10,"Sandstrom, Miss. Marguerite Rut",1,1
13,"Andersson, Mr. Anders Johan",1,5
16,"Rice, Master. Eugene",4,1
24,"Palsson, Miss. Torborg Danira",3,1
...,...,...,...
856,"Wick, Mrs. George Dennick (Mary Hitchcock)",1,1
863,"Sage, Miss. Dorothy Edith ""Dolly""",8,2
869,"Johnson, Master. Harold Theodor",1,1
871,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",1,1
